In [ ]:
import jsonimport pandas as pdimport timeimport reimport astimport requestsimport shutilfrom urllib.parse import urljoin, quotefrom bs4 import BeautifulSoup

In [ ]:
# df view settingspd.set_option('display.max_colwidth', None)pd.set_option('display.max_columns', None)

In [ ]:
# HTTP session setup (avoids Selenium)DEFAULT_HEADERS = {    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "                  "(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",    "Accept-Language": "en-US,en;q=0.9",}session = requests.Session()session.headers.update(DEFAULT_HEADERS)def fetch_soup(url: str) -> BeautifulSoup:    """Fetch a page and parse it with BeautifulSoup."""    resp = session.get(url, timeout=30)    resp.raise_for_status()    return BeautifulSoup(resp.text, "html.parser")

In [ ]:
# retrieving all the distinct car brandsu = "https://www.bilbasen.dk/brugt/bil?includeengroscvr=true&includeleasing=false"data = json.loads(fetch_soup(u).find("script", id="__NEXT_DATA__").string)c = []def walk(x):    if isinstance(x, list):        labels = []        for v in x:            if isinstance(v, str):                labels.append(v.strip())            elif isinstance(v, dict):                for k in ("label", "name", "title", "text", "value", "displayName"):                    s = v.get(k)                    if isinstance(s, str):                        labels.append(s.strip())                        break        if len(labels) >= 30:            uniq = sorted(set(labels))            good = [                s for s in uniq                if s and len(s) <= 30 and s[0].isalpha() and s[0].isupper()                and not any(ch.isdigit() for ch in s)            ]            if len(good) / len(uniq) > 0.8:                c.append(uniq)        for v in x:            walk(v)    elif isinstance(x, dict):        for v in x.values():            walk(v)walk(data)car_brands = sorted(c, key=len, reverse=True)[1]

In [ ]:
fuel_options = {    1: 'Benzin',    2: 'Diesel',    3: 'El',    6: 'Hybrid - Benzin',    8: 'Hybrid - Diesel',    11: 'Plug-in Benzin',    12: 'Plug-in Diesel'}

In [ ]:
def download_brand(brand: str, selected_fuel_type: str):    """Collect listing URLs for a brand without Selenium."""    page_listings = []    brand_slug = quote(brand.strip(), safe="")    base_url = (        f"https://www.bilbasen.dk/brugt/bil/{brand_slug}"        f"?fuel={selected_fuel_type}&includeengroscvr=true&includeleasing=false"    )    soup = fetch_soup(base_url)    page_tag = soup.find('span', {'data-e2e': 'pagination-total'})    if not (page_tag and page_tag.text.isdigit()):        print(f"→ Skipping {brand}: 0 pages found")        return None    max_page = int(page_tag.text)    for page in range(1, max_page + 1):        paged_url = base_url if page == 1 else f"{base_url}&page={page}"        if page > 1:            soup = fetch_soup(paged_url)        print(f"Fetching {brand} page {page}/{max_page}")        for art in soup.find_all("article"):            if "".join(art.get("class", [])).startswith("Listing_listing"):                for a in art.find_all("a", class_=lambda c: c and c.startswith("Listing_link")):                    href = a.get("href")                    if href:                        page_listings.append(urljoin("https://www.bilbasen.dk", href))    return page_listingslistings = []for b in car_brands:    pl = download_brand(b, str(selected_fuel_type))    if pl:        listings.extend(pl)

In [ ]:
print(len(listings), len(set(listings)))

In [ ]:
# Getting JSON data from each listing page (avoid navigating tag hierarchies). ~ 1 minute per 100 carsall_parsed_data = []total = len(set(listings))last_report = time.time()def parse_listing_page(link: str):    json_text = None    soup = fetch_soup(link)    for s in soup.find_all("script"):        txt = (s.get_text() or "").lstrip()        if txt.startswith("var _props"):            m = re.search(r"var\s*_props\s*=\s*({.*?})\s*;", txt, flags=re.DOTALL)            if m:                json_text = m.group(1)                break    if not json_text:        print("No _props JSON found on this page " + link)        return None    try:        return json.loads(json_text)    except Exception as e:        print("Error parsing JSON:", e)        return Nonedef func_wrapper_for_loop(i, link):    global last_report    # Progress monitoring after each 100 pages    if i % 100 == 0 or i == total:        now = time.time()        elapsed = now - last_report        mins, secs = divmod(int(elapsed), 60)        print(            f"{i}/{total} listings done "            f"({i/total:.1%}) — last batch took {mins}m {secs}s",            flush=True        )        last_report = now    parsed_data = parse_listing_page(link)    if parsed_data:        all_parsed_data.append(parsed_data)for i, link in enumerate(set(listings), start=1):    func_wrapper_for_loop(i, link)

In [ ]:
# digest messy JSON data into a flat table of readable datadef extract_name_value(row):    output = {}    # Iterate over each cell in the row with its column label.    for col, cell in row.items():        # If the cell is a dictionary with the desired keys, transform it.        if isinstance(cell, dict) and 'name' in cell and 'displayValue' in cell:            output[cell['name']] = cell['displayValue']        # If the cell is a string, try to parse it.        elif isinstance(cell, str):            try:                d = ast.literal_eval(cell)                if isinstance(d, dict) and 'name' in d and 'displayValue' in d:                    output[d['name']] = d['displayValue']                else:                    # Not the desired structure, so keep the original cell under its column name.                    output[col] = cell            except Exception:                # Parsing failed; keep the original cell.                output[col] = cell        else:            # For any other type, simply keep the original cell.            output[col] = cell    return pd.Series(output)

In [ ]:
all_listings = []for entry in all_parsed_data:    # try old key    listing_data = entry.get("listing")    # fall back to new path    if listing_data is None:        listing_data = []        for q in (            entry.get("props", {})                 .get("pageProps", {})                 .get("dehydratedState", {})                 .get("queries", [])        ):            listing_data.extend(q.get("state", {}).get("data", {}).get("listings", []))    if listing_data:        # keep one level of nesting: 'vehicle.modelInformation' stays a dict        flat = pd.json_normalize(listing_data, sep=".", max_level=1)        all_listings.append(flat)all_listings = pd.concat(all_listings, ignore_index=True)

In [ ]:
# Unpacking nested dictionaries into separate columnsdf_model_info = all_listings['vehicle.modelInformation'].apply(pd.Series)df_vehicle_details = all_listings['vehicle.details'].apply(pd.Series)df_ratings = all_listings['vehicle.ratings.subRatings'].apply(pd.Series)df_base = all_listings.drop(['vehicle.modelInformation', 'vehicle.details', 'vehicle.ratings.subRatings'], axis=1)df_expanded = pd.concat([df_base, df_model_info, df_vehicle_details], axis=1)

In [ ]:
rows = [extract_name_value(row) for _, row in df_expanded.iterrows()]df_result = pd.DataFrame(rows)

In [ ]:
benzin_cols = [        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering',         'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',        'Drivmiddel', 'Brændstofforbrug','Cylindre', 'Airbags', 'Tankkapacitet','ABS-bremser', 'ESP', 'Periodisk afgift','CO2 udledning', 'Euronorm',         'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description']el_cols = [        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering',         'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',        'Drivmiddel', 'Energiforbrug', 'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC', 'Hurtig opladning DC', 'Opladningstid DC 10-80%',        'Airbags', 'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift',         'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description'    ]

In [ ]:
columns = {    'Benzin': benzin_cols,    'Diesel': benzin_cols,    'El':     el_cols,}

In [ ]:
today = pd.Timestamp.now().replace(microsecond=0)yesterday = today - pd.Timedelta(days=1)print(today, yesterday)df_result.insert(0, 'scrape_timestamp', today)

In [ ]:
df = df_result[columns[fuel_options[selected_fuel_type]]].copy()

In [ ]:
today_str = today.strftime("%Y-%m-%d")df.to_parquet(    f"/home/pi-vault/projects/bilbasen_webscraping/data/{fuel_options[selected_fuel_type]}/"    f"{fuel_options[selected_fuel_type]}_listings_{today_str}.parquet",    index=True,    engine="fastparquet",)